In [2]:
# Cell 0: One-time package installation
"""
Install the packages needed to connect local Python/Jupyter to Bloomberg.

Run this cell once, then restart the kernel:
Kernel > Restart Kernel
"""

%pip install --upgrade pandas openpyxl xbbg

%pip install --upgrade --index-url=https://blpapi.bloomberg.com/repository/releases/python/simple/ blpapi

Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://blpapi.bloomberg.com/repository/releases/python/simple/
Note: you may need to restart the kernel to use updated packages.


In [1]:
# Cell 1: Confirm local Python environment
"""
This confirms that you are running Jupyter locally.

For Bloomberg Desktop API access, this should show Windows,
not Linux/AWS/Anaconda Cloud.
"""

import sys
import platform
import struct

print("Python version:", sys.version)
print("Python executable:", sys.executable)
print("Platform:", platform.platform())
print("Machine:", platform.machine())
print("64-bit Python:", struct.calcsize("P") * 8 == 64)

Python version: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
Python executable: C:\Users\apasten\AppData\Local\anaconda3\python.exe
Platform: Windows-11-10.0.22631-SP0
Machine: AMD64
64-bit Python: True


In [8]:
# Cell 2: Import packages and define helper functions
"""
Import packages and create helper functions.

Bloomberg/xbbg may return pandas, Narwhals, or PyArrow-style tables.
The helper functions below convert Bloomberg output into normal pandas DataFrames.
"""

import pandas as pd
import blpapi
from xbbg import blp
from pathlib import Path

print("Packages imported successfully.")


def to_pandas_df(obj):
    """
    Convert Bloomberg/xbbg output into a normal pandas DataFrame.
    """

    # Already pandas
    if isinstance(obj, pd.DataFrame):
        return obj.copy()

    # Narwhals object -> native object
    if hasattr(obj, "to_native"):
        obj = obj.to_native()

    # Native object may still be pandas
    if isinstance(obj, pd.DataFrame):
        return obj.copy()

    # PyArrow table -> pandas
    if hasattr(obj, "to_pandas"):
        return obj.to_pandas()

    # Last-resort conversion
    return pd.DataFrame(obj)


def clean_bdh_output(raw_output):
    """
    Clean Bloomberg historical data from blp.bdh().

    Final output columns:
    date | bloomberg_identifier | field | value
    """

    df = to_pandas_df(raw_output)

    # Older xbbg versions may return MultiIndex columns
    if isinstance(df.columns, pd.MultiIndex):
        df = df.copy()
        df.index.name = "date"
        df.columns = df.columns.set_names(["bloomberg_identifier", "field"])

        long_df = (
            df
            .stack(level="bloomberg_identifier")
            .reset_index()
        )

        # Convert field columns into long format
        long_df = long_df.melt(
            id_vars=["date", "bloomberg_identifier"],
            var_name="field",
            value_name="value"
        )

    else:
        # Newer xbbg output often comes as:
        # ticker | date | field | value
        df = df.copy()
        df.columns = [str(col).lower() for col in df.columns]

        long_df = df.rename(columns={
            "ticker": "bloomberg_identifier"
        })

    # Standardize date
    if "date" in long_df.columns:
        long_df["date"] = pd.to_datetime(long_df["date"])

    # Standardize order
    keep_cols = ["date", "bloomberg_identifier", "field", "value"]
    long_df = long_df[keep_cols]

    return long_df


def clean_bdp_output(raw_output):
    """
    Clean Bloomberg static data from blp.bdp().

    Final output is wide:
    bloomberg_identifier | field_1 | field_2 | ...
    """

    df = to_pandas_df(raw_output)
    df = df.copy()

    # Newer xbbg output often comes as:
    # ticker | field | value
    df.columns = [str(col).lower() for col in df.columns]

    if {"ticker", "field", "value"}.issubset(df.columns):
        df = df.rename(columns={"ticker": "bloomberg_identifier"})

        wide_df = (
            df
            .pivot_table(
                index="bloomberg_identifier",
                columns="field",
                values="value",
                aggfunc="first"
            )
            .reset_index()
        )

        wide_df.columns.name = None

    else:
        # Fallback for older wide format
        wide_df = df.reset_index().rename(columns={
            "index": "bloomberg_identifier"
        })

    return wide_df

Packages imported successfully.


In [9]:
# Cell 3: Test Bloomberg connection
"""
This confirms that local Python can communicate with the open Bloomberg Terminal.
"""

test_raw = blp.bdp(
    tickers="SPX Index",
    flds=["PX_LAST"]
)

test = clean_bdp_output(test_raw)

display(test)
print(type(test))

,bloomberg_identifier,PX_LAST
0,SPX Index,7398.93


<class 'pandas.DataFrame'>


In [10]:
# Cell 4: Test one FRL 6460 project index
"""
This confirms that one of the project Bloomberg index identifiers works.
"""

project_test_raw = blp.bdh(
    tickers=["I03840US Index"],
    flds=["PX_LAST"],
    start_date="2018-12-31",
    end_date="2019-01-10"
)

project_test = clean_bdh_output(project_test_raw)

display(project_test)
print(type(project_test))

,date,bloomberg_identifier,field,value
0,2018-12-31,I03840US Index,PX_LAST,177.0761
1,2019-01-02,I03840US Index,PX_LAST,177.0720
2,2019-01-03,I03840US Index,PX_LAST,177.5543
3,2019-01-04,I03840US Index,PX_LAST,177.1482
4,2019-01-07,I03840US Index,PX_LAST,177.0387
5,2019-01-08,I03840US Index,PX_LAST,176.8362
6,2019-01-09,I03840US Index,PX_LAST,176.9268
7,2019-01-10,I03840US Index,PX_LAST,176.9370


<class 'pandas.DataFrame'>


In [11]:
# Cell 5: Project dashboard
"""
Dashboard for the FRL 6460 Bloomberg index download.

Edit this cell if you need to change:
1. Index identifiers
2. Date range
3. Fields
4. Output file name
"""

index_map = {
    "Short Treasury / 1-4Y": {
        "bbg_id": "I03840US Index",
        "long_name": "Bloomberg U.S. Treasury: 1-4 Yr Total Return Index"
    },
    "Intermediate Treasury / 3-7Y": {
        "bbg_id": "LT13TRUU Index",
        "long_name": "Bloomberg U.S. Treasury 3-7 Yr Total Return Index"
    },
    "Core Treasury / 3-10Y": {
        "bbg_id": "LT31TRUU Index",
        "long_name": "Bloomberg U.S. Treasury 3-10 Year Total Return Index"
    },
    "Investment Grade Long Credit": {
        "bbg_id": "I02787US Index",
        "long_name": "Bloomberg US Long Credit Aaa Total Return Index"
    },
    "High Yield Workbook Label / Baa Long Credit": {
        "bbg_id": "I02788US Index",
        "long_name": "Bloomberg US Long Credit Baa Total Return Index"
    }
}

index_ids = [info["bbg_id"] for info in index_map.values()]

start_date = "2018-12-31"
end_date = "2025-12-31"

historical_fields = ["PX_LAST"]

static_fields = [
    "SECURITY_NAME",
    "CRNCY"
]

download_static_data = True
download_historical_data = True
calculate_daily_returns = True
export_to_excel = True

output_file = "FRL6460_Bloomberg_Index_Data_2018_2025.xlsx"

In [12]:
# Cell 6: Create index reference table
"""
Create a clean table linking workbook labels to Bloomberg identifiers.
"""

index_reference = pd.DataFrame([
    {
        "workbook_label": label,
        "bloomberg_identifier": info["bbg_id"],
        "long_name": info["long_name"]
    }
    for label, info in index_map.items()
])

display(index_reference)

,workbook_label,bloomberg_identifier,long_name
0,Short Treasury / 1-4Y,I03840US Index,Bloomberg U.S. Treasury: 1-4 Yr Total Return I...
1,Intermediate Treasury / 3-7Y,LT13TRUU Index,Bloomberg U.S. Treasury 3-7 Yr Total Return Index
2,Core Treasury / 3-10Y,LT31TRUU Index,Bloomberg U.S. Treasury 3-10 Year Total Return...
3,Investment Grade Long Credit,I02787US Index,Bloomberg US Long Credit Aaa Total Return Index
4,High Yield Workbook Label / Baa Long Credit,I02788US Index,Bloomberg US Long Credit Baa Total Return Index


In [13]:
# Cell 7: Download static Bloomberg information
"""
Static data checks that Bloomberg recognizes each index identifier.
"""

static_data = None

if download_static_data:
    print("Downloading static data...")

    static_raw = blp.bdp(
        tickers=index_ids,
        flds=static_fields
    )

    static_data = clean_bdp_output(static_raw)

    static_data = index_reference.merge(
        static_data,
        on="bloomberg_identifier",
        how="left"
    )

    display(static_data)

else:
    print("Static data switch is OFF.")

,workbook_label,bloomberg_identifier,long_name,CRNCY,SECURITY_NAME
0,Short Treasury / 1-4Y,I03840US Index,Bloomberg U.S. Treasury: 1-4 Yr Total Return I...,USD,Bloomberg U.S. Treasury: 1-4 Y
1,Intermediate Treasury / 3-7Y,LT13TRUU Index,Bloomberg U.S. Treasury 3-7 Yr Total Return Index,USD,Bloomberg U.S. Treasury 3-7 Yr
2,Core Treasury / 3-10Y,LT31TRUU Index,Bloomberg U.S. Treasury 3-10 Year Total Return...,USD,Bloomberg U.S. Treasury 3-10 Y
3,Investment Grade Long Credit,I02787US Index,Bloomberg US Long Credit Aaa Total Return Index,USD,Bloomberg US Long Credit Aaa T
4,High Yield Workbook Label / Baa Long Credit,I02788US Index,Bloomberg US Long Credit Baa Total Return Index,USD,Bloomberg US Long Credit Baa T


In [14]:
# Cell 8: Download historical total return index levels
"""
Download historical total return index levels from Bloomberg.

PX_LAST is the main field because these are total return indexes.
"""

historical_long = None

if download_historical_data:
    print("Downloading historical index levels...")

    historical_raw = blp.bdh(
        tickers=index_ids,
        flds=historical_fields,
        start_date=start_date,
        end_date=end_date
    )

    historical_long = clean_bdh_output(historical_raw)

    historical_long = historical_long.merge(
        index_reference,
        on="bloomberg_identifier",
        how="left"
    )

    historical_long = historical_long[
        [
            "date",
            "workbook_label",
            "bloomberg_identifier",
            "long_name",
            "field",
            "value"
        ]
    ]

    display(historical_long.head())
    display(historical_long.tail())

else:
    print("Historical data switch is OFF.")

,date,workbook_label,bloomberg_identifier,long_name,field,value
0,2018-12-31,Short Treasury / 1-4Y,I03840US Index,Bloomberg U.S. Treasury: 1-4 Yr Total Return I...,PX_LAST,177.0761
1,2019-01-02,Short Treasury / 1-4Y,I03840US Index,Bloomberg U.S. Treasury: 1-4 Yr Total Return I...,PX_LAST,177.0720
2,2019-01-03,Short Treasury / 1-4Y,I03840US Index,Bloomberg U.S. Treasury: 1-4 Yr Total Return I...,PX_LAST,177.5543
3,2019-01-04,Short Treasury / 1-4Y,I03840US Index,Bloomberg U.S. Treasury: 1-4 Yr Total Return I...,PX_LAST,177.1482
4,2019-01-07,Short Treasury / 1-4Y,I03840US Index,Bloomberg U.S. Treasury: 1-4 Yr Total Return I...,PX_LAST,177.0387


,date,workbook_label,bloomberg_identifier,long_name,field,value
8880,2025-12-25,High Yield Workbook Label / Baa Long Credit,I02788US Index,Bloomberg US Long Credit Baa Total Return Index,PX_LAST,6239.4039
8881,2025-12-26,High Yield Workbook Label / Baa Long Credit,I02788US Index,Bloomberg US Long Credit Baa Total Return Index,PX_LAST,6232.3772
8882,2025-12-29,High Yield Workbook Label / Baa Long Credit,I02788US Index,Bloomberg US Long Credit Baa Total Return Index,PX_LAST,6244.7124
8883,2025-12-30,High Yield Workbook Label / Baa Long Credit,I02788US Index,Bloomberg US Long Credit Baa Total Return Index,PX_LAST,6237.7183
8884,2025-12-31,High Yield Workbook Label / Baa Long Credit,I02788US Index,Bloomberg US Long Credit Baa Total Return Index,PX_LAST,6226.3498


In [15]:
# Cell 9: Calculate daily returns
"""
Calculate daily returns from the total return index level.

Because these are total return indexes, daily percentage changes in PX_LAST
represent the daily total return for each index.
"""

historical_returns_long = None

if calculate_daily_returns and historical_long is not None:

    index_levels_long = historical_long[
        historical_long["field"] == "PX_LAST"
    ].copy()

    index_levels_long = index_levels_long.rename(columns={
        "value": "index_level"
    })

    index_levels_long["index_level"] = pd.to_numeric(
        index_levels_long["index_level"],
        errors="coerce"
    )

    index_levels_long = index_levels_long.sort_values(
        ["bloomberg_identifier", "date"]
    )

    index_levels_long["daily_return"] = (
        index_levels_long
        .groupby("bloomberg_identifier")["index_level"]
        .pct_change()
    )

    historical_returns_long = index_levels_long

    display(historical_returns_long.head(10))

else:
    print("Daily return calculation is OFF or historical data is missing.")

,date,workbook_label,bloomberg_identifier,long_name,field,index_level,daily_return
5331,2018-12-31,Investment Grade Long Credit,I02787US Index,Bloomberg US Long Credit Aaa Total Return Index,PX_LAST,3014.1616,NaN
5332,2019-01-02,Investment Grade Long Credit,I02787US Index,Bloomberg US Long Credit Aaa Total Return Index,PX_LAST,3027.2147,0.004331
5333,2019-01-03,Investment Grade Long Credit,I02787US Index,Bloomberg US Long Credit Aaa Total Return Index,PX_LAST,3052.5635,0.008374
5334,2019-01-04,Investment Grade Long Credit,I02787US Index,Bloomberg US Long Credit Aaa Total Return Index,PX_LAST,3025.9171,-0.008729
5335,2019-01-07,Investment Grade Long Credit,I02787US Index,Bloomberg US Long Credit Aaa Total Return Index,PX_LAST,3034.2549,0.002755
5336,2019-01-08,Investment Grade Long Credit,I02787US Index,Bloomberg US Long Credit Aaa Total Return Index,PX_LAST,3031.9312,-0.000766
5337,2019-01-09,Investment Grade Long Credit,I02787US Index,Bloomberg US Long Credit Aaa Total Return Index,PX_LAST,3032.8640,0.000308
5338,2019-01-10,Investment Grade Long Credit,I02787US Index,Bloomberg US Long Credit Aaa Total Return Index,PX_LAST,3016.4312,-0.005418
5339,2019-01-11,Investment Grade Long Credit,I02787US Index,Bloomberg US Long Credit Aaa Total Return Index,PX_LAST,3024.4755,0.002667
5340,2019-01-14,Investment Grade Long Credit,I02787US Index,Bloomberg US Long Credit Aaa Total Return Index,PX_LAST,3013.7865,-0.003534


In [16]:
# Cell 10: Create wide tables
"""
Create wide tables that look closer to Excel.

These are useful for comparing index levels and returns across indexes.
"""

index_levels_wide = None
daily_returns_wide = None

if historical_returns_long is not None:

    index_levels_wide = historical_returns_long.pivot(
        index="date",
        columns="workbook_label",
        values="index_level"
    )

    daily_returns_wide = historical_returns_long.pivot(
        index="date",
        columns="workbook_label",
        values="daily_return"
    )

    display(index_levels_wide.head())
    display(daily_returns_wide.head())

else:
    print("No return data available.")

workbook_label,Core Treasury / 3-10Y,High Yield Workbook Label / Baa Long Credit,Intermediate Treasury / 3-7Y,Investment Grade Long Credit,Short Treasury / 1-4Y
date,,,,,
2018-12-31,199.4281,4868.0802,373.32,3014.1616,177.0761
2019-01-02,199.6337,4884.5379,373.56,3027.2147,177.0720
2019-01-03,200.9483,4919.1386,375.80,3052.5635,177.5543
2019-01-04,199.7958,4895.8137,373.88,3025.9171,177.1482
2019-01-07,199.5057,4920.9422,373.36,3034.2549,177.0387


workbook_label,Core Treasury / 3-10Y,High Yield Workbook Label / Baa Long Credit,Intermediate Treasury / 3-7Y,Investment Grade Long Credit,Short Treasury / 1-4Y
date,,,,,
2018-12-31,NaN,NaN,NaN,NaN,NaN
2019-01-02,0.001031,0.003381,0.000643,0.004331,-0.000023
2019-01-03,0.006585,0.007084,0.005996,0.008374,0.002724
2019-01-04,-0.005735,-0.004742,-0.005109,-0.008729,-0.002287
2019-01-07,-0.001452,0.005133,-0.001391,0.002755,-0.000618


In [17]:
# Cell 11: Export results to Excel
"""
Export all downloaded and cleaned Bloomberg data to Excel.
"""

if export_to_excel:

    print("Exporting data to Excel...")

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

        index_reference.to_excel(
            writer,
            sheet_name="Index_Reference",
            index=False
        )

        if static_data is not None:
            static_data.to_excel(
                writer,
                sheet_name="Static_Data",
                index=False
            )

        if historical_long is not None:
            historical_long.to_excel(
                writer,
                sheet_name="Historical_Long",
                index=False
            )

        if historical_returns_long is not None:
            historical_returns_long.to_excel(
                writer,
                sheet_name="Historical_Returns_Long",
                index=False
            )

        if index_levels_wide is not None:
            index_levels_wide.to_excel(
                writer,
                sheet_name="Index_Levels_Wide"
            )

        if daily_returns_wide is not None:
            daily_returns_wide.to_excel(
                writer,
                sheet_name="Daily_Returns_Wide"
            )

    print(f"Saved file: {output_file}")

else:
    print("Export switch is OFF. No Excel file created.")

Exporting data to Excel...
Saved file: FRL6460_Bloomberg_Index_Data_2018_2025.xlsx
